# 036 — Proyecto: sistema híbrido para decisiones

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia en 5 minutos

**El proyecto integra las tres familias de la Parte 02** en un sistema de decisión con capas separadas y contratos explícitos:

```text
evidencia e ──> [Red bayesiana]  P(H|e)          (creencias, clases 025-027)
P(H|e) + U ──> [Decisión]        a* = argmax EU  (utilidad esperada, clase 030;
                                                  política MDP si es secuencial, 029)
θ libres  ──> [Evolución]        θ* sobre escenarios Monte Carlo (031, 033-034)
todo      ──> [Trazabilidad]     evidencia, posterior, EU, acción, semilla
```

**Criterios de diseño:** (1) separar estimación (`P`) de preferencia (`U`) — el costo del error vive en `U`, nunca inflando probabilidades; (2) matriz de utilidades completa `U(acción, estado)`; (3) **análisis de sensibilidad**: encontrar el *valor de cruce* del parámetro donde la acción óptima cambia — si está lejos de la estimación, la decisión es robusta; (4) umbral de abstención cuando la ventaja de `a*` no supera el ruido del estimador.

**Evaluación honesta:** la utilidad esperada estimada por Monte Carlo lleva error ~1/√N; comparar dos configuraciones exige semillas comunes o N suficiente para que la diferencia supere el error estándar.


### Mini ejemplo

Triaje de mantenimiento (del README): prior de falla 0.10, sensor con TPR 0.9 / FPR 0.2, se observa vibración → posterior `1/3`. Utilidades: detener = −5 fijo; continuar = −40 si falla, 0 si no. `EU(detener) = −5 > EU(continuar) = −13.3` → detener. Valor de cruce: la decisión se mantiene mientras el posterior supere `0.125`. El laboratorio `capstone` encadena exactamente estas capas con traza JSON.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("capstone", seed=36)
show(result)


## Reflexión

1. En la traza del laboratorio, identifica qué campo pertenece a cada capa (creencia, decisión, optimización, trazabilidad). ¿Falta alguna capa? ¿Qué añadirías?
2. ¿Por qué "ajustar el prior para ser más precavidos" es peor diseño que "ajustar la utilidad del falso negativo"? ¿Qué se vuelve inauditable en el primer caso?
3. Propón el análisis de sensibilidad mínimo que exigirías antes de dejar que este sistema decida sin revisión humana, y el criterio cuantitativo de abstención.
